# Phase D-1: FNO Distillation

## PINN → FNO Surrogate 증류

```
Trained PINN (느림, ~50ms/query)
    ↓ 10,000쌍 데이터 생성
    ↓ (설계변수 → PSF 7개)
FNO Surrogate (빠름, ~1ms/query)
    ↓
BoTorch가 이 FNO를 사용하여 역설계
```

### 왜 FNO가 필요한가?
```
PINN:  한 번 예측에 ~50ms (autograd 포함)
FNO:   한 번 예측에 ~1ms (forward pass만)
       → 50배 빠름

BoTorch 역설계에서 수천 번 호출
  PINN: 50ms × 5000 = 250초
  FNO:  1ms × 5000 = 5초   ← 이것이 "8초 역설계"의 핵심
```

In [ ]:
import sys, json, time
from pathlib import Path

def _find_root():
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / 'pyproject.toml').exists(): return p
        p = p.parent
    raise FileNotFoundError('Cannot find project root')

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

DEVICE = torch.device('cpu')
print('Ready')

---
## Step 1: PINN에서 FNO 학습 데이터 생성

학습된 PINN에 다양한 설계변수를 넣어서 PSF를 수집합니다.

In [ ]:
from backend.core.pinn_model import PurePINN
from backend.physics.psf_metrics import compute_psf_7
from backend.training.loss_functions import ASMIncidentLUT
from backend.training.warm_start import warm_start
from backend.training.loss_functions import helmholtz_loss, phase_loss, bm_boundary_loss
from backend.training.collocation_sampler import hierarchical_collocation
from backend.training.curriculum import CurriculumConfig, get_loss_weights

# ── Train PINN (10K collocation) or load existing ──
ckpt_path = PROJECT_ROOT / 'checkpoints' / 'phase_c_final.pt'
if not ckpt_path.exists():
    ckpt_path = PROJECT_ROOT / 'checkpoints' / 'phase_c_cpu_10k.pt'

if ckpt_path.exists():
    print(f'Loading PINN: {ckpt_path.name}')
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    cfg = ckpt.get('config', {'hidden_dim': 64, 'num_layers': 3, 'num_freqs': 24, 'omega_0': 30.0})
    model = PurePINN(**cfg).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
else:
    print('No checkpoint. Training PINN (10K colloc, 3000 ep)...')
    torch.manual_seed(42)
    model = PurePINN(hidden_dim=64, num_layers=3, num_freqs=24).to(DEVICE)
    asm_lut = ASMIncidentLUT(str(PROJECT_ROOT / 'data' / 'asm_luts' / 'incident_z40.npz'))
    warm_start(model, asm_lut, epochs=100, n_points=5000, device=DEVICE, log_every=50)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3000, eta_min=1e-5)
    curriculum = CurriculumConfig(total_epochs=3000)
    for epoch in range(3000):
        model.train()
        w = get_loss_weights(epoch, curriculum)
        coords = hierarchical_collocation(10000, DEVICE)
        L_H = helmholtz_loss(model, coords) if w['lambda_H'] > 0 else torch.tensor(0.0)
        L_ph = phase_loss(model, asm_lut, 500, DEVICE)
        L_bc = bm_boundary_loss(model, 500, DEVICE)
        L = w['lambda_H']*L_H + w['lambda_phase']*L_ph + w['lambda_BC']*L_bc
        optimizer.zero_grad(); L.backward(); optimizer.step(); scheduler.step()
        if epoch % 1000 == 0: print(f'  Epoch {epoch}: L={L.item():.4e}')
    print('Training done')

model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'PINN: {n_params:,} params')

In [ ]:
# ── Generate 10,000 (design_params → PSF) pairs ──
N = 10000
np.random.seed(42)

params = np.stack([
    np.random.uniform(-10, 10, N),   # delta_bm1
    np.random.uniform(-10, 10, N),   # delta_bm2
    np.random.uniform(5, 20, N),     # w1
    np.random.uniform(5, 20, N),     # w2
    np.random.uniform(0, 40, N),     # theta
], axis=1).astype(np.float32)

psfs = np.zeros((N, 7), dtype=np.float32)
t0 = time.time()
for i in range(N):
    psfs[i] = compute_psf_7(model, *params[i,:4], params[i,4], DEVICE, n_samples=100)
    if (i+1) % 2000 == 0:
        print(f'  {i+1}/{N} ({time.time()-t0:.0f}s)')

# Normalize
psf_max = psfs.max()
if psf_max > 0:
    psfs_norm = psfs / psf_max
else:
    psfs_norm = psfs

save_path = PROJECT_ROOT / 'data' / 'fno_training' / 'pinn_distill_data.npz'
save_path.parent.mkdir(parents=True, exist_ok=True)
np.savez(save_path, params=params, psfs=psfs_norm)

print(f'\nGenerated {N} pairs in {time.time()-t0:.0f}s')
print(f'PSF range (raw): [{psfs.min():.2e}, {psfs.max():.2e}]')
print(f'PSF range (norm): [{psfs_norm.min():.4f}, {psfs_norm.max():.4f}]')
print(f'Saved: {save_path.name}')

---
## Step 2: FNO Distillation

In [ ]:
# Run distillation script
print('FNO Distillation (CLI):')
print('  python scripts/distill_fno.py --epochs 500')
print()

# Or check existing result
fno_result_path = PROJECT_ROOT / 'experiments' / 'fno_distillation.json'
if fno_result_path.exists():
    with open(fno_result_path) as f:
        fno_r = json.load(f)
    print('Existing FNO result:')
    print(f'  Val loss:      {fno_r["best_val_loss"]:.6f}')
    print(f'  Inference:     {fno_r["inference_ms"]:.1f} ms')
    print(f'  Val error:     {fno_r["val_rel_error_pct"]:.1f}%')
    print(f'  Training time: {fno_r["training_time_s"]/60:.0f} min')
else:
    print('No FNO result yet. Run distill_fno.py first.')

---
## Step 3: FNO Accuracy Check

In [ ]:
fno_ckpt = PROJECT_ROOT / 'checkpoints' / 'fno_surrogate.pt'
if fno_ckpt.exists():
    from backend.core.fno_model import FNOSurrogate
    
    ckpt_fno = torch.load(fno_ckpt, map_location=DEVICE, weights_only=False)
    fno = FNOSurrogate().to(DEVICE)
    fno.load_state_dict(ckpt_fno['model_state_dict'])
    fno.eval()
    p_mean = ckpt_fno['p_mean']
    p_std = ckpt_fno['p_std']
    
    # Test on validation data
    data = np.load(PROJECT_ROOT / 'data' / 'fno_training' / 'pinn_distill_data.npz')
    test_params = torch.from_numpy(data['params'][-100:])
    test_psfs = data['psfs'][-100:]
    
    test_norm = (test_params - p_mean) / p_std
    with torch.no_grad():
        pred = fno(test_norm.float()).numpy()
    
    # Compare PINN vs FNO
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, i in zip(axes.flat, range(6)):
        ax.bar(range(7), test_psfs[i], width=0.35, label='PINN', alpha=0.7, color='#42A5F5')
        ax.bar(np.arange(7)+0.35, pred[i], width=0.35, label='FNO', alpha=0.7, color='#EF5350')
        ax.set_xticks(range(7))
        ax.set_xticklabels(['R','V','R','V(c)','R','V','R'], fontsize=7)
        p = data['params'][-100+i]
        ax.set_title(f'd1={p[0]:+.0f} w1={p[2]:.0f} th={p[4]:.0f}', fontsize=8)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.2, axis='y')
    
    plt.suptitle('FNO vs PINN: PSF Comparison', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Speed test
    t0 = time.time()
    for _ in range(1000):
        with torch.no_grad(): _ = fno(test_norm[:1].float())
    ms_per = (time.time() - t0) / 1000 * 1000
    print(f'FNO inference: {ms_per:.2f} ms/sample')
else:
    print('No FNO checkpoint. Run distill_fno.py first.')